In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt

# Run this notebook from the repository root.
DATA_PATH = (
    "data/"
    "PMCOA_samples_01234_overlap_clean_split_headers.tsv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Run this notebook from the repository root."
    )

df = pd.read_csv(DATA_PATH, sep="\t")

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
df.head(10)

In [ ]:
valid_labels = {"I", "M", "R", "D", "NoIMRaD"}

assert set(df["IMRaD"].dropna().unique()).issubset(valid_labels)
assert df["split"].notna().all()

labels_per_section = (
    df.groupby(["pmcid", "section_id"])["IMRaD"].nunique()
)
assert labels_per_section.max() == 1

section_level = (
    df.sort_values(["pmcid", "section_id", "section_position", "sentence_id"])
      .drop_duplicates(["pmcid", "section_id"])
      .copy()
)

print("Articles:", section_level["pmcid"].nunique())
print("Section blocks:", len(section_level))

print("\nSection labels:")
print(section_level["IMRaD"].value_counts())

print("\nSection blocks by split:")
print(section_level["split"].value_counts())

print("\nSection labels by split:")
print(pd.crosstab(section_level["IMRaD"], section_level["split"]))

In [ ]:
import matplotlib.pyplot as plt

section_level["section_id"] = pd.to_numeric(
    section_level["section_id"],
    errors="raise"
)

section_level = section_level.sort_values(["pmcid", "section_id"])

layouts = (
    section_level.groupby("pmcid")
    .agg(
        split=("split", "first"),
        layout=("IMRaD", lambda x: "→".join(x)),
        n_blocks=("IMRaD", "size")
    )
    .reset_index()
)

print("Blocks per article:")
print(layouts["n_blocks"].describe())

print("\nUnique layouts:", layouts["layout"].nunique())

print("\nMost frequent layouts:")
print(layouts["layout"].value_counts().head(20))

print("\nUnique layouts by split:")
print(layouts.groupby("split")["layout"].nunique())

In [ ]:
# dataset summary

dataset_summary = pd.DataFrame({
    "articles": layouts.groupby("split")["pmcid"].nunique(),
    "section_blocks": section_level.groupby("split").size()
}).reindex(["train", "val", "test"])

dataset_summary.loc["overall"] = [
    layouts["pmcid"].nunique(),
    len(section_level)
]

dataset_summary["article_percent"] = (
    dataset_summary["articles"]
    / dataset_summary.loc["overall", "articles"]
    * 100
).round(2)

dataset_summary["block_percent"] = (
    dataset_summary["section_blocks"]
    / dataset_summary.loc["overall", "section_blocks"]
    * 100
).round(2)

print(dataset_summary)